# Evidencia reproducible del dataset supervisado

**Pregunta que responde:** si el *modeling dataset* canónico puede materializarse y aceptarse de forma reproducible para un símbolo, método de etiquetado y taxonomía dados.

**Evidencia que deja:**
- entradas usadas (`features` y `event_labels`),
- ejecución del *workflow* oficial,
- checks de aceptación y *readiness*,
- rutas y hashes de los artefactos persistidos,
- resumen del dataset resultante,
- plan *walk-forward* generado.

**No cubre:**
- distribuciones globales, *missingness* o correlaciones del *feature set* (`02_features/01_eda_features_intermedias.ipynb`),
- comparación metodológica de métodos de etiquetado (`02_features/02_evaluacion_etiquetado_targets.ipynb`),
- protocolo de selección de variables *intra-fold* (`02_features/03_metodologia_seleccion_features.ipynb`),
- preexperimento FFD o screening de columnas (`04_preexperimento/`),
- entrenamiento, shortlists ni evidencia de estudios (`05_modelado_predictivo/`).

Este notebook no regenera *labels* desde OHLCV: exige `event_labels_4h.parquet` ya materializado y falla explícitamente si falta.

**Productos:** informe de aceptación (`supervised_dataset_acceptance_report.json`, `supervised_dataset_acceptance_checks.csv`, `supervised_dataset_acceptance_report_manifest.json`) bajo `reports/validation/supervised/<logical_dataset_name>/dataset_acceptance/`; *parquets* de `modeling_dataset_4h` y `event_labels_4h` bajo `data/03_processed/datasets/` y `data/03_processed/targets/` (rutas exactas en las tablas de la última celda). No exporta figuras.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def locate_repo_root(start: Path | None = None) -> Path:
    root = (start or Path.cwd()).resolve()
    while root != root.parent and not (root / "src").exists():
        root = root.parent
    if not (root / "src").exists():
        raise FileNotFoundError("No se encontró la raíz del repo (carpeta src/).")
    return root


ROOT = locate_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config.settings import Settings
from src.targets.contract import TargetMethod
from src.targets.supervised_dataset_workflow import (
    build_target_config_from_method,
    run_supervised_dataset_workflow,
)
from src.validation.walk_forward import WalkForwardConfig, WalkForwardMode


In [2]:
SYMBOL = "BTCUSDT"
METHOD = TargetMethod.FIXED_HORIZON
TAXONOMY = "technical"

FEATURES_ROOT = ROOT / "data" / "02_intermediate"
EVENT_LABELS_ROOT = ROOT / "data" / "03_processed"
PROCESSED_ROOT = ROOT / "data" / "03_processed"
REPORT_ROOT = ROOT / "reports" / "validation" / "supervised"

settings = Settings.load_from_yaml(ROOT / "config" / "settings.yaml")
target_config = build_target_config_from_method(
    METHOD,
    horizon_bars=6,
    vertical_horizon_bars=6,
    max_scan_bars=24,
)

wf_defaults = settings.validation.walk_forward
walk_forward_config = WalkForwardConfig.from_target_config(
    target_config,
    mode=WalkForwardMode(wf_defaults.mode),
    train_bars=wf_defaults.train_bars,
    test_bars=wf_defaults.test_bars,
    step_bars=wf_defaults.step_bars,
    min_train_bars=wf_defaults.min_train_bars,
    purge_bars_override=max(wf_defaults.purge_bars, target_config.required_purge_bars),
    embargo_bars_override=max(wf_defaults.embargo_bars, target_config.required_embargo_bars),
)

features_path = FEATURES_ROOT / f"{SYMBOL.lower()}_features_4h.parquet"
labels_path = EVENT_LABELS_ROOT / "targets" / SYMBOL / METHOD.value / "event_labels_4h.parquet"

if not features_path.exists():
    raise FileNotFoundError(f"No existe el parquet de features: {features_path}")
if not labels_path.exists():
    raise FileNotFoundError(
        "No existe el parquet de labels. Este notebook requiere event_labels_4h.parquet ya materializado.\n"
        f"Ruta esperada: {labels_path}"
    )


In [3]:
workflow = run_supervised_dataset_workflow(
    features=features_path,
    event_labels=labels_path,
    symbol=SYMBOL,
    target_config=target_config,
    taxonomy=TAXONOMY,
    processed_root=PROCESSED_ROOT,
    report_root=REPORT_ROOT,
    repo_root=ROOT,
    walk_forward_config=walk_forward_config,
)

summary = workflow.to_dict()
quality_stats = (summary.get("quality_report") or {}).get("stats") or {}
model_readiness = summary.get("model_readiness") or {}
walk_forward = summary.get("walk_forward_plan") or {}
acceptance = summary.get("acceptance_outcome") or {}

run_summary = pd.DataFrame(
    [
        {
            "dataset": summary.get("logical_dataset_name") or summary.get("dataset_name"),
            "símbolo": summary.get("symbol"),
            "método": summary.get("method"),
            "aceptado": acceptance.get("accepted"),
            "listo_para_entrenar": acceptance.get("ready_for_training"),
            "checks_calidad": (summary.get("quality_report") or {}).get("checks_run"),
            "checks_readiness": model_readiness.get("checks_run"),
            "filas": quality_stats.get("label_total"),
            "distribución_clases": quality_stats.get("class_distribution"),
            "n_folds": walk_forward.get("n_folds"),
            "rango_dataset": walk_forward.get("dataset_range"),
        }
    ]
)

display(run_summary)

2026-06-04 02:48:43 [info     ] build_start                    component=supervised_builder dataset_name=technical_fixed_horizon_4h_v1 method=fixed_horizon symbol=BTCUSDT taxonomy=technical
2026-06-04 02:48:43 [info     ] inputs_loaded                  component=supervised_builder features_shape=(18707, 127) labels_shape=(18844, 17) symbol=BTCUSDT
2026-06-04 02:48:43 [info     ] columns_selected               active_columns=29 available_features=127 component=supervised_builder selected_features=29 symbol=BTCUSDT
2026-06-04 02:48:43 [info     ] join_complete                  component=supervised_builder features_rows=18707 joined_rows=18707 labels_rows=18844 symbol=BTCUSDT
2026-06-04 02:48:44 [info     ] quality_gate_passed            component=supervised_builder summary=SupervisedQualityGate PASSED: 17 checks, 0 violations, 0 warnings symbol=BTCUSDT
2026-06-04 02:48:44 [info     ] build_complete                 component=supervised_builder dataset_name=technical_fixed_horizon_4h_v1 fe

,dataset,símbolo,método,aceptado,listo_para_entrenar,checks_calidad,checks_readiness,filas,distribución_clases,n_folds,rango_dataset
0,btcusdt_supervised_fixed_horizon_4h_v1_core_only,BTCUSDT,fixed_horizon,True,True,17,11,18707,"{-1: 7031, 0: 3939, 1: 7737}",97,2017-08-17 04:00:00 -> 2026-02-28 20:00:00


In [4]:
checks_df = pd.DataFrame(workflow.acceptance_checks)
checks_view = (
    checks_df[checks_df["check_id"] != "DSA_AUDIT_TRIAL_TRACEABILITY"]
    .assign(
        required=lambda d: d["required"].map({True: "sí", False: "no"}),
        blocking=lambda d: d["blocking"].map({True: "sí", False: "no"}),
    )[["check_id", "status", "required", "blocking", "summary"]]
    .rename(
        columns={
            "check_id": "id_check",
            "status": "estado",
            "required": "requerido",
            "blocking": "bloqueante",
            "summary": "descripción",
        }
    )
    .reset_index(drop=True)
)

display(checks_view)

,id_check,estado,requerido,bloqueante,descripción
0,DSA_STRUCT_DATASET_MATERIALIZED,PASS,sí,no,Integridad: dataset materializado reproducible...
1,DSA_STRUCT_TARGET_RUN_METADATA,PASS,sí,no,Integridad: metadatos de run de targets cohere...
2,DSA_STRUCT_TEMPORAL_COLUMNS,PASS,sí,no,Integridad: columnas temporales obligatorias p...
3,DSA_STRUCT_VERSIONS_HASHES,PASS,sí,no,Integridad: versiones y hashes de trazabilidad...
4,DSA_BUILDER_QUALITY_GATE,PASS,sí,no,Quality gate del builder supervisado superado
5,DSA_MODEL_READINESS,PASS,sí,no,Readiness temporal/solapamiento/walk-forward (...
6,DSA_WF_PURGE_EMBARGO_CONTRACT,PASS,sí,no,Walk-forward: purge_bars y embargo_bars >= mín...
7,DSA_WF_PLAN_VALIDATED,PASS,sí,no,Walk-forward: plan purged validado sin violaci...
8,DSA_WF_INTERVAL_LEAKAGE,PASS,sí,no,Walk-forward: ausencia de leakage intervalar t...
9,DSA_WF_EMBARGO_INTER_FOLD,PASS,sí,no,Walk-forward: embargo efectivo entre folds con...


## Cómo leer el informe de aceptación

La tabla anterior (`checks_df`) es la **lectura abreviada** del bloque `acceptance_checks` del informe canónico persistido por el *workflow* en:

```
reports/validation/supervised/<logical_dataset_name>/dataset_acceptance/
├── supervised_dataset_acceptance_report.json
├── supervised_dataset_acceptance_checks.csv
└── supervised_dataset_acceptance_report_manifest.json
```

El JSON conserva los nombres del contrato (no se traducen) porque son identificadores estables del *bundle* auditable. Bloques clave:

- `report_kind`, `report_version`, `generated_at`, `report_sha256`: identidad y huella del propio informe.
- `verdict`, `ready_for_training`: veredicto global y autorización operativa para entrenar.
- `dataset_identity.*`: rango temporal, número de filas, hashes de columnas activas y rutas del *modeling dataset*.
- `temporal_contract.*`: convención causal (`decision_ts`, `execution_ts`, *purge* y *embargo*) que el plan *walk-forward* debe respetar.
- `experiment_spec.*`: enlace con el diseño experimental (`logical_dataset_name`, hashes de identidad y política de selección).
- `modeling_readiness.*`: resultado agregado de los *gates* de calidad y *model readiness*.
- `artifact_index.*`: rutas relativas y SHA-256 de los artefactos consumidos (*parquets* y `target_run.json`).
- `tracking.*`: `config_hash` y, si aplica, `trial_id` para anclar la corrida a un *trial* de un estudio.
- `acceptance_checks`: lista detallada equivalente a la tabla `checks_df` de arriba.

El informe en disco es el artefacto que se cita en memoria; `checks_df` lo resume en formato tabular legible. La tabla mostrada filtra el check opcional `DSA_AUDIT_TRIAL_TRACEABILITY`, que en esta corrida no aplica (la trazabilidad por trial/registry se exige solo cuando el dataset se materializa anclado a un estudio).

> **Nota sobre los avisos `embargo_inter_fold_advisory`** que pueden aparecer en el log del *workflow*: corresponden a folds consecutivos cuyo *test* siguiente arranca justo dentro de la ventana de embargo del anterior. Son **advertencias informativas**, no violaciones del contrato del fold actual: el check `DSA_WF_EMBARGO_INTER_FOLD` evalúa el embargo *intra-fold* del plan validado y queda en **PASS** mientras `purge_bars` y `embargo_bars` operativos cumplan el mínimo contractual del `TargetConfig` (ver fila `DSA_WF_PURGE_EMBARGO_CONTRACT`).

## Artefactos persistidos

El workflow escribe el paquete auditable bajo `reports/validation/supervised/<logical_dataset_name>/dataset_acceptance/` y conserva los hashes/rutas en el informe canónico. La celda siguiente extrae la información necesaria para citar la corrida: rutas de aceptación, manifiesto de artefactos, resumen del dataset y plan de folds.


In [5]:
import json

acceptance_paths = pd.DataFrame(
    [
        {"artefacto": name, "ruta": path}
        for name, path in (summary.get("acceptance_report_paths") or {}).items()
    ]
)
display(acceptance_paths)

artifact_rows = []
for check in workflow.acceptance_checks:
    detail = check.get("detail") or {}
    if isinstance(detail, str):
        try:
            detail = json.loads(detail)
        except json.JSONDecodeError:
            detail = {}
    manifest = detail.get("artifact_manifest") or {}
    for artifact_name, artifact_info in manifest.items():
        if isinstance(artifact_info, dict):
            artifact_rows.append(
                {
                    "artefacto": artifact_name,
                    "ruta": artifact_info.get("relative_path") or artifact_info.get("path"),
                    "sha256": artifact_info.get("sha256"),
                }
            )

if artifact_rows:
    display(pd.DataFrame(artifact_rows).drop_duplicates())
else:
    print("[aviso] No se encontró artifact_manifest en los checks de aceptación.")

modeling_df = workflow.build_result.modeling_dataset.copy()
modeling_df["decision_ts"] = pd.to_datetime(modeling_df["decision_ts"])

dataset_summary = pd.DataFrame(
    [
        {
            "filas": len(modeling_df),
            "columnas": modeling_df.shape[1],
            "decision_ts_min": modeling_df["decision_ts"].min(),
            "decision_ts_max": modeling_df["decision_ts"].max(),
            "target_no_nulo": modeling_df["target_label"].notna().sum(),
            "valores_target": sorted(int(v) for v in modeling_df["target_label"].dropna().unique()),
        }
    ]
)
display(dataset_summary)

fold_rows = []
if workflow.walk_forward_plan is not None:
    for fold in workflow.walk_forward_plan.folds:
        fold_rows.append(
            {
                "fold": fold.fold_id,
                "n_train": fold.n_train,
                "n_test": fold.n_test,
                "inicio_train": fold.train_start,
                "fin_train": fold.train_end,
                "inicio_test": fold.test_start,
                "fin_test": fold.test_end,
            }
        )
folds_df = pd.DataFrame(fold_rows)
if not folds_df.empty:
    display(folds_df.head(5))
    display(folds_df.tail(5))

,artefacto,ruta
0,report_json,/app/reports/validation/supervised/btcusdt_sup...
1,report_csv,/app/reports/validation/supervised/btcusdt_sup...
2,manifest_json,/app/reports/validation/supervised/btcusdt_sup...


,artefacto,ruta,sha256
0,event_labels_4h,data/03_processed/targets/BTCUSDT/fixed_horizo...,6a9f1ab96e635d0585cdf24b69b677026cabf96bdd87d3...
1,modeling_dataset_4h,data/03_processed/datasets/BTCUSDT/fixed_horiz...,829696787bd5c7b29b293e756d75fdd261a75d0a0b20f5...


,filas,columnas,decision_ts_min,decision_ts_max,target_no_nulo,valores_target
0,18707,57,2017-08-17 04:00:00,2026-02-28 20:00:00,18707,"[-1, 0, 1]"


,fold,n_train,n_test,inicio_train,fin_train,inicio_test,fin_test
0,0,1080,180,2017-08-17 04:00:00,2018-02-13,2018-02-14 04:00:00,2018-03-16
1,1,1080,180,2017-09-16 04:00:00,2018-03-15,2018-03-16 04:00:00,2018-04-15
2,2,1080,180,2017-10-16 04:00:00,2018-04-14,2018-04-15 04:00:00,2018-05-15
3,3,1080,180,2017-11-15 04:00:00,2018-05-14,2018-05-15 04:00:00,2018-06-14
4,4,1080,180,2017-12-15 04:00:00,2018-06-13,2018-06-14 04:00:00,2018-07-14


,fold,n_train,n_test,inicio_train,fin_train,inicio_test,fin_test
92,92,1080,180,2025-03-08 04:00:00,2025-09-04,2025-09-05 04:00:00,2025-10-05
93,93,1080,180,2025-04-07 04:00:00,2025-10-04,2025-10-05 04:00:00,2025-11-04
94,94,1080,180,2025-05-07 04:00:00,2025-11-03,2025-11-04 04:00:00,2025-12-04
95,95,1080,180,2025-06-06 04:00:00,2025-12-03,2025-12-04 04:00:00,2026-01-03
96,96,1080,180,2025-07-06 04:00:00,2026-01-02,2026-01-03 04:00:00,2026-02-02
